# ML-32M Dataset Cleaning

## Overview
This notebook documents the cleaning process for the MovieLens 32M dataset.

## Dataset Info
- **Dataset**: MovieLens 32M (ml-32m)
- **Source**: MovieLens/GroupLens Research
- **Size**: ~32 million ratings

## Cleaning Steps

### 1. Initial Data Loading
- Load ratings, movies, tags and links data
- Check basic structure and dimensions

### 2. Data Quality Checks
- Missing values
- Duplicate entries


### 3. Cleaning Operations
- Remove duplicates
- Handle missing data
- Fix formatting issues
- Validate data types

### 4. Data Validation
- Verify all movieIds have corresponding movie metadata
- Check rating ranges (0.5 to 5.0)

### 5. Output
- Save cleaned dataset
- Generate cleaning report/statistics

---

### Importing the Libraries & Loading the Datasets

In [45]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [46]:
pd.set_option('display.float_format', '{:.2f}'.format)


ERROR! Session/line number was not unique in database. History logging moved to new session 297


In [47]:
df_ratings = pd.read_csv("../datasets/ratings.csv")
df_movies  = pd.read_csv("../datasets/movies.csv")
df_tags    = pd.read_csv("../datasets/tags.csv")
df_links = pd.read_csv('../datasets/links.csv')


- Checking the data

In [48]:
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.00,944249077
1,1,25,1.00,944250228
2,1,29,2.00,943230976
3,1,30,5.00,944249077
4,1,32,5.00,943228858


In [49]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [50]:
df_tags.head()

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [51]:
df_links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.00
1,2,113497,8844.00
2,3,113228,15602.00
3,4,114885,31357.00
4,5,113041,11862.00


---



### Data Quality Checks & Cleaning Operations

In [52]:
df_ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [53]:
df_tags.isna().sum()

userId        0
movieId       0
tag          17
timestamp     0
dtype: int64

In [54]:
df_movies.isna().sum()

movieId    0
title      0
genres     0
dtype: int64

In [55]:
df_links.isna().sum()

movieId      0
imdbId       0
tmdbId     124
dtype: int64

In [56]:
df_links.dropna(inplace= True)
df_tags.dropna(inplace=True)

In [57]:
df_links.isna().sum()

movieId    0
imdbId     0
tmdbId     0
dtype: int64

In [58]:
df_tags.isna().sum()

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

Checking for duplicates

In [59]:
df_ratings.duplicated().sum()

np.int64(0)

In [60]:
df_movies.duplicated().sum()

np.int64(0)

In [61]:
df_tags.duplicated().sum()

np.int64(0)

In [62]:

df_links.duplicated().sum()

np.int64(0)

Checking the datatypes

In [63]:
df_ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

In [64]:
df_movies.dtypes

movieId     int64
title      object
genres     object
dtype: object

In [65]:
df_tags.dtypes

userId        int64
movieId       int64
tag          object
timestamp     int64
dtype: object

Checking data ranges in the dataframes, we will onnly check ratings as ther other 2 dataframes barely hold any numerical information other than the id/timestamp

In [66]:
df_ratings.describe()

,userId,movieId,rating,timestamp
count,32000204.00,32000204.00,32000204.00,32000204.00
mean,100278.51,29318.61,3.54,1275241199.57
std,57949.05,50958.16,1.06,256162975.95
min,1.00,1.00,0.50,789652004.00
25%,50053.00,1233.00,3.00,1051012430.00
50%,100297.00,3452.00,3.50,1272621812.50
75%,150451.00,44199.00,4.00,1503158345.25
max,200948.00,292757.00,5.00,1697164147.00


The rating distribution range is valid (0-5)

- Converting all timestamps to actual dates

In [67]:
df_ratings['timestamp'] = pd.to_datetime(df_ratings['timestamp'],unit='s')
df_tags['timestamp'] = pd.to_datetime(df_tags['timestamp'],unit='s')


- decreasing the loading time of the rating dataset onto the database and for a better recommendation performance

Keeping the top 30% active users

In [68]:
users_activity = df_ratings.groupby('userId')['rating'].count().reset_index() 
valid_users = users_activity.loc[users_activity['rating'] > users_activity['rating'].quantile(0.7), 'userId']

df_ratings = df_ratings.loc[df_ratings['userId'].isin(valid_users)]
df_tags = df_tags.loc[df_tags['userId'].isin(valid_users)]

print(f'Ratings DF size: {df_ratings.shape}')
print(f'Tags DF size: {df_tags.shape}')

Ratings DF size: (23930291, 4)
Tags DF size: (1832382, 4)


Keeping the 30% most rated movies

In [69]:
df_validCount = df_ratings.groupby('movieId')['rating'].count().reset_index()
filt = df_validCount['rating'] > float(df_validCount['rating'].quantile(0.7))
movies_needed = df_validCount.loc[filt, 'movieId']

df_ratings = df_ratings.loc[df_ratings['movieId'].isin(movies_needed)]
df_movies = df_movies.loc[df_movies['movieId'].isin(movies_needed)]
df_tags = df_tags.loc[df_tags['movieId'].isin(movies_needed)]
df_links = df_links.loc[df_links['movieId'].isin(movies_needed)]

In [70]:
print(f'Movies DF size: {df_movies.shape}')
print(f'Ratings DF size: {df_ratings.shape}')
print(f'Tags DF size: {df_tags.shape}')

Movies DF size: (24484, 3)
Ratings DF size: (23689983, 4)
Tags DF size: (1665375, 4)


In [71]:
# drop duplicates based on the primary key columns only
df_tags = df_tags.drop_duplicates(subset=['movieId','userId','tag'])

---

### Feature Scaling

In this step, we will add the weighted average rating for each movie

In [72]:
rating_stats = df_ratings.groupby('movieId')['rating'].agg(
    v='count',
    R='mean'
).reset_index()

C = rating_stats['R'].mean()          # global mean rating
m = rating_stats['v'].quantile(0.70)  # minimum vote threshold (70th percentile)

rating_stats['weighted_rating'] = (
    (rating_stats['v'] / (rating_stats['v'] + m)) * rating_stats['R'] +
    (m / (rating_stats['v'] + m)) * C
)

df_movies = df_movies.merge(rating_stats[['movieId', 'weighted_rating']], on='movieId', how='left')



---

### Data Cleaned

Data is clean now, we can start by loading it into a database.
- Saving the file
- Loading the data into a database (MYSQL)

In [73]:
df_tags.to_csv('../datasets/cleaned_datasets/tags_cleaned.csv', index = False)
df_movies.to_csv('../datasets/cleaned_datasets/movies_cleaned.csv', index = False)
df_ratings.to_csv('../datasets/cleaned_datasets/ratings_cleaned.csv', index = False)
df_links.to_csv('../datasets/cleaned_datasets/links_cleaned.csv', index = False)

---

